# Tutorial: the six Project 1 functions

This notebook is a **learning notes** version of `implementations.py`.

You will see, for each method:

1. What problem it solves
2. The formula (same convention as CS-433 / the public tests)
3. How the NumPy code maps to that formula
4. A tiny numerical check

The graded file is the copy at the **repo root**: `implementations.py`.
Read this notebook, then re-type the code yourself. Copying without understanding will not help on the exam.

**Rules from the project PDF**

- Return `(w, loss)` only for the **last** `w`.
- `w` is 1D: shape `(D,)`, not `(D, 1)`.
- MSE uses a factor **`1/(2N)`**.
- SGD mini-batch size is **1**.
- Logistic labels are **`{0, 1}`** (not `{+1, -1}`).
- Ridge / regularized logistic: **returned loss does not include** $\lambda\|w\|^2$.
- Do not use `numpy.linalg.lstsq`.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

## 0. Shared notation

| Symbol | Meaning | NumPy |
|---|---|---|
| $N$ | number of samples | `tx.shape[0]` |
| $D$ | number of features (including a column of 1s if you add a bias) | `tx.shape[1]` |
| $y\in\mathbb{R}^N$ | targets | `y` shape `(N,)` |
| $X\in\mathbb{R}^{N\times D}$ | design matrix | `tx` |
| $w\in\mathbb{R}^D$ | weights | `w` shape `(D,)` |
| $e = y - Xw$ | residual | `e = y - tx @ w` |

Prediction for a linear model with two features (this toy `tx`):

$$\hat{y}_n = w_0 x_{n0} + w_1 x_{n1}$$

In matrix form that is $\hat{y} = Xw$ (`tx @ w`). Sample $n$ uses **row** $n$ of $X$.

The public tests use this toy data (keep it in mind while you read):

In [ ]:
y = np.array([0.1, 0.3, 0.5])
tx = np.array([[2.3, 3.2], [1.0, 0.1], [1.4, 2.3]])
initial_w = np.array([0.5, 1.0])
gamma = 0.1
max_iters = 2
print("N, D =", tx.shape)

N, D = (3, 2)


## 1. Mean squared error (the loss for the first four methods)

$$
L(w) = \frac{1}{2N}\|y - Xw\|^2 = \frac{1}{2N}\sum_{n=1}^N e_n^2
$$

The extra $1/2$ is there so the $2$ cancels when we differentiate. **Do not drop it** — the tests check the number.

Gradient (vector form from the lab):

$$
\nabla L(w) = -\frac{1}{N} X^\top e = \frac{1}{N} X^\top (Xw - y)
$$

In [3]:
def mse(y, tx, w):
    e = y - tx @ w
    return 0.5 * np.mean(e**2)


def mse_gradient(y, tx, w):
    e = y - tx @ w
    return -(tx.T @ e) / y.shape[0]


print("L(initial_w) =", mse(y, tx, initial_w))
print("grad          =", mse_gradient(y, tx, initial_w))

L(initial_w) = 4.067083333333334
grad          = [4.525 6.46 ]


## 2. `mean_squared_error_gd` — linear regression + gradient descent

**Idea.** Walk downhill: at every step, move a little opposite the gradient.

$$
w \leftarrow w - \gamma\,\nabla L(w)
$$

`gamma` is the step size. Too large $\gamma$ can diverge; too small is slow.

If `max_iters == 0`, do **no** update: return the starting `w` and $L(w)$.

In [4]:
def mean_squared_error_gd(y, tx, initial_w, max_iters, gamma):
    w = np.asarray(initial_w, dtype=float).reshape(-1)
    y = np.asarray(y).reshape(-1)
    for _ in range(max_iters):
        w = w - gamma * mse_gradient(y, tx, w)
    return w, mse(y, tx, w)


w, loss = mean_squared_error_gd(y, tx, initial_w, max_iters, gamma)
print(w, loss)
print("expected w    [ -0.050586,  0.203718 ]")
print("expected loss 0.051534")

[-0.050586  0.203718] 0.051533911025167085
expected w    [ -0.050586,  0.203718 ]
expected loss 0.051534


## 3. `mean_squared_error_sgd` — same loss, one random point per step

Full GD uses **all** $N$ rows in the gradient. SGD uses **one** row $n$:

$$
w \leftarrow w - \gamma\,\nabla L_n(w)
$$

In code, that is the same `mse_gradient` function called on `y[n:n+1]` and `tx[n:n+1]` (so $N=1$ in the formula).

Why SGD? One step is cheap when $N$ is 300k. The path is noisier.

The tests pass a dataset of length 1, so the “random” index is always 0.

In [5]:
def mean_squared_error_sgd(y, tx, initial_w, max_iters, gamma):
    w = np.asarray(initial_w, dtype=float).reshape(-1)
    y = np.asarray(y).reshape(-1)
    n = y.shape[0]
    for _ in range(max_iters):
        i = np.random.randint(n)
        w = w - gamma * mse_gradient(y[i : i + 1], tx[i : i + 1], w)
    return w, mse(y, tx, w)


w, loss = mean_squared_error_sgd(y[:1], tx[:1], initial_w, max_iters, gamma)
print(w, loss)
print("expected w    [ 0.063058,  0.39208 ]")
print("expected loss 0.844595")

[0.063058 0.39208 ] 0.8445947735940318
expected w    [ 0.063058,  0.39208 ]
expected loss 0.844595


## 4. `least_squares` — closed form (normal equations)

Set $\nabla L(w)=0$:

$$
X^\top X\, w = X^\top y \qquad\Rightarrow\qquad w^\star = (X^\top X)^{-1} X^\top y
$$

Do **not** invert with `np.linalg.inv` if you can avoid it. Solve the linear system:

```python
w = np.linalg.solve(tx.T @ tx, tx.T @ y)
```

`lstsq` is forbidden in this project. Loss is still MSE with $1/(2N)$.

In [6]:
def least_squares(y, tx):
    y = np.asarray(y).reshape(-1)
    w = np.linalg.solve(tx.T @ tx, tx.T @ y)
    return w, mse(y, tx, w)


w, loss = least_squares(y, tx)
print(w, loss)
print("expected w    [ 0.218786, -0.053837 ]")
print("expected loss 0.026942")

[ 0.218786 -0.053837] 0.026941580756013744
expected w    [ 0.218786, -0.053837 ]
expected loss 0.026942


## 5. `ridge_regression` — least squares + $\lambda\|w\|^2$

We minimize

$$
L_{\text{ridge}}(w) = \underbrace{\frac{1}{2N}\|y-Xw\|^2}_{\text{MSE}} + \lambda\|w\|^2.
$$

Set the gradient to 0:

$$
\frac{1}{N} X^\top(Xw-y) + 2\lambda w = 0
\quad\Rightarrow\quad
(X^\top X + 2N\lambda I)\, w = X^\top y.
$$

The **$2N$** comes from mixing MSE’s $1/(2N)$ with a penalty $\lambda\|w\|^2$ (not $\frac{\lambda}{2N}\|w\|^2$). The public tests use this convention.

Returned `loss` is **plain MSE**, without $\lambda\|w\|^2$.

If $\lambda=0$, ridge **is** least squares.

In [8]:
def ridge_regression(y, tx, lambda_):
    y = np.asarray(y).reshape(-1)
    n, d = tx.shape
    a = tx.T @ tx + 2 * n * lambda_ * np.eye(d)
    w = np.linalg.solve(a, tx.T @ y)
    return w, mse(y, tx, w)


print("lambda=0", ridge_regression(y, tx, 0.0))
print("lambda=1", ridge_regression(y, tx, 1.0))
print("expected lambda=1 w [ 0.054303, 0.042713 ], loss 0.03175")
print(tx.shape)

lambda=0 (array([ 0.218786, -0.053837]), np.float64(0.026941580756013744))
lambda=1 (array([0.054303, 0.042713]), np.float64(0.031749586214239095))
expected lambda=1 w [ 0.054303, 0.042713 ], loss 0.03175
(3, 2)


## 6. Logistic regression — binary classification

Linear regression outputs any real number. Classification needs a probability in $(0,1)$.

$$
\sigma(z) = \frac{1}{1+e^{-z}}, \qquad
p(y=1\mid x) = \sigma(x^\top w).
$$

**Labels must be 0 and 1.** The competition uses `{+1,-1}`; convert before calling this function:
`y01 = (y + 1) / 2`.

Loss = mean negative log-likelihood:

$$
L(w) = -\frac{1}{N}\sum_{n=1}^N \Big[
 y_n \log \sigma(x_n^\top w) + (1-y_n)\log\big(1-\sigma(x_n^\top w)\big)
\Big].
$$

Gradient:

$$
\nabla L(w) = \frac{1}{N} X^\top \big(\sigma(Xw) - y\big).
$$

Then the same GD loop as MSE: $w \leftarrow w - \gamma\nabla L(w)$.

In [9]:
def sigmoid(z):
    z = np.clip(z, -30, 30)
    return 1.0 / (1.0 + np.exp(-z))


def logistic_loss(y, tx, w):
    p = np.clip(sigmoid(tx @ w), 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


def logistic_gradient(y, tx, w):
    return (tx.T @ (sigmoid(tx @ w) - y)) / y.shape[0]


def logistic_regression(y, tx, initial_w, max_iters, gamma):
    y = np.asarray(y, dtype=float).reshape(-1)
    w = np.asarray(initial_w, dtype=float).reshape(-1)
    for _ in range(max_iters):
        w = w - gamma * logistic_gradient(y, tx, w)
    return w, logistic_loss(y, tx, w)


y_bin = (y > 0.2) * 1.0  # same conversion as the public tests
w, loss = logistic_regression(y_bin, tx, initial_w, max_iters, gamma)
print(w, loss)
print("expected w    [ 0.378561,  0.801131 ]")
print("expected loss 1.348358")

[0.378561 0.801131] 1.348357817512931
expected w    [ 0.378561,  0.801131 ]
expected loss 1.348358


## 7. `reg_logistic_regression` — logistic + $\lambda\|w\|^2$

$$
L_{\text{reg}}(w) = L_{\text{NLL}}(w) + \lambda\|w\|^2
\qquad\Rightarrow\qquad
\nabla L_{\text{reg}} = \nabla L_{\text{NLL}} + 2\lambda w.
$$

GD uses the **regularized** gradient. The value you **return** is $L_{\text{NLL}}$ only (no penalty), as required by the PDF.

Large $\lambda$ pulls $w$ toward 0 (simpler model, less overfitting).

In [10]:
def reg_logistic_regression(y, tx, lambda_, initial_w, max_iters, gamma):
    y = np.asarray(y, dtype=float).reshape(-1)
    w = np.asarray(initial_w, dtype=float).reshape(-1)
    for _ in range(max_iters):
        grad = logistic_gradient(y, tx, w) + 2.0 * lambda_ * w
        w = w - gamma * grad
    return w, logistic_loss(y, tx, w)


w, loss = reg_logistic_regression(y_bin, tx, 1.0, initial_w, max_iters, gamma)
print(w, loss)
print("expected w    [ 0.216062,  0.467747 ]")
print("expected loss 0.972165")

[0.216062 0.467747] 0.9721649929512525
expected w    [ 0.216062,  0.467747 ]
expected loss 0.972165


## 8. How the six methods fit together

```text
continuous y          |  GD on MSE     -> mean_squared_error_gd
                      |  SGD on MSE    -> mean_squared_error_sgd
                      |  exact MSE     -> least_squares
                      |  exact MSE+L2  -> ridge_regression
----------------------+----------------------------------------
binary y in {0,1}     |  GD on NLL     -> logistic_regression
                      |  GD on NLL+L2  -> reg_logistic_regression
```

On the heart-disease data you will usually:

1. Map labels `{+1,-1}` $\to$ `{0,1}` for logistic.
2. Standardize columns of `tx`.
3. Prefer **logistic** / **ridge** over plain least squares for classification.
4. Choose $\lambda$ and $\gamma$ with **cross-validation**, not the AIcrowd leaderboard.

Next: open `implementations.py`, cover it, and rewrite each function from memory. Then run the public tests from `grading_tests/` against a **local path** to this repo.